In [2]:
!pip install sentence-transformers faiss-cpu

In [3]:
!pip install pypdf

In [15]:
!pip install -r requirements.txt

  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
Using cached torch-2.10.0-cp313-cp313-win_amd64.whl (113.8 MB)


In [16]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

In [17]:
config_data = json.load(open("config.json"))
HF_TOKEN = config_data["HF TOKEN"]

In [18]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [19]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


In [20]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)


In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

In [22]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    token=HF_TOKEN
)

Loading weights: 100%|██████████| 201/201 [00:25<00:00,  7.91it/s, Materializing param=model.norm.weight]                              


In [23]:
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [24]:
def get_response(prompt):
    output = text_generator(prompt)
    return output[0]["generated_text"]

In [25]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Rohan_J\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohan_J\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1111.33it/s, Ma

In [27]:
from pypdf import PdfReader

def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

doc = load_pdf("NIPS-2017-attention-is-all-you-need-Paper.pdf")
def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks
chunks = chunk_text(doc)

In [28]:
chunk_embeddings = embed_model.encode(chunks)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

In [29]:
def retrieve(query, top_k=3):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    return [chunks[i] for i in indices[0]]

In [41]:
def build_prompt(query, context_chunks):
    context = "\n\n".join(context_chunks)
    prompt = f"""
You are a strict QA system.

Answer ONLY using the information provided in the CONTEXT below.
If the answer is not explicitly stated in the context, respond exactly with:

I don't know.

Do not guess.
Do not use prior knowledge.
Do not explain beyond the context."

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [42]:
def rag_answer(query):
    retrieved_chunks = retrieve(query)
    final_prompt = build_prompt(query, retrieved_chunks)
    return get_response(final_prompt)

In [44]:
print(rag_answer("What is Regularization?"))

Regularization is a technique used in machine learning to prevent overfitting, which is the
occasion when the model learns patterns that are not present in the training data. Regularization
means adding a term to the loss function to penalize the model for overfitting. It is a technique
used to prevent the model from overfitting to the training data.
